## Sintezavimo sprendimo instrukcija

Žemiau pateiktos techninės instrukcijos ir skriptai, skirtos lietuviškos TTS sistemos administratoriui, kurie parodo, kaip galima kurti lokaliai (vietiniame kompiuteryje) veikiantį TTS sprendimą, skirtą sintezuoti lietuvišką šneką. Vietiniame kompiuteryje reikia turėti:
1. Akustinio modelio archyvą
2. Vokoderio archyvą
   
### 0. Inicializuokite pagalbines funkcijas

In [ ]:
# Inicializuokite pagalbines funkcijas

from espnet2.bin.tts_inference import Text2Speech
import soundfile as sf
from IPython.display import Audio, HTML
from espnet2.main_funcs.pack_funcs import unpack, find_path_and_change_it_recursive
import tarfile
from pathlib import Path
from typing import Dict
import yaml

cache_dir = "./.cache/"
device = "cpu"
#device = "cuda" # uncomment if you have a GPU and want to use it

def _safe_extract_tar(tar: tarfile.TarFile, path: Path):
    base = path.resolve()

    for member in tar.getmembers():
        member_path = (path / member.name).resolve()
        if not str(member_path).startswith(str(base)):
            raise Exception("Unsafe tar archive")

    tar.extractall(path)

def find_vocoder(search_dir: Path) -> str:
    for file in search_dir.rglob("*.pkl"):
        # print(f"found {str(file)}")
        return str(file)

    raise FileNotFoundError(f"No *.pkl file found in the {search_dir}")    


def get_dict_from_cache(meta: Path) -> Dict[str, str]:
    if not meta.exists():
        raise FileNotFoundError(f"No meta.yaml found in the {meta}")
        
    outpath = meta.parent
    with meta.open("r", encoding="utf-8") as f:
        d = yaml.safe_load(f)
        yaml_files = d["yaml_files"]
        files = d["files"]

        for _, value in yaml_files.items():
            if not (outpath / value).exists():
                raise FileNotFoundError(f"No {outpath / value} found")
            with (outpath / value).open("r", encoding="utf-8") as f:
                d_int = yaml.safe_load(f)
            for path in outpath.rglob("*"):
                if path.is_file():
                    relative_path = path.relative_to(outpath)
                    d_int = find_path_and_change_it_recursive(d_int, str(relative_path), str(path))    
            with (outpath / value).open("w", encoding="utf-8") as f:
                yaml.safe_dump(d_int, f)
        retval = {}
        for key, value in list(yaml_files.items()) + list(files.items()):
            if not (outpath / value).exists():
                raise FileNotFoundError(f"No {outpath / value} found")
            retval[key] = str(outpath / value)
        return retval
        

def extract_archive_cached(archive_path: str, cache_dir: str) -> Path:
    archive_path = Path(archive_path)
    if archive_path.is_dir():
        return archive_path  # already extracted
    
    cache_dir = Path(cache_dir)

    stat = archive_path.stat()

    # fast cache key (no hashing)
    key = f"{archive_path.name}_{stat.st_size}_{int(stat.st_mtime)}"

    extract_path = cache_dir / key
    marker = extract_path / ".done"

    if marker.exists():
        return extract_path

    extract_path.mkdir(parents=True, exist_ok=True)

    if archive_path.name.endswith(".tar.gz"):
        with tarfile.open(archive_path) as tar:
            _safe_extract_tar(tar, extract_path)
    else:
        raise ValueError(f"Unsupported archive format: {archive_path}")

    marker.touch()
    return extract_path
    

def init_tts(am_file=None, vocoder_file=None):
    if not vocoder_file:
        vocoder_file = None # make sure it's None, not empty string
    else:
        vocoder_dir = extract_archive_cached(archive_path=vocoder_file, cache_dir=cache_dir)
        vocoder_file = find_vocoder(vocoder_dir)
        print(f"vocoder_file: {vocoder_file}")

    print(f"\n===================================\nAM      = {am_file}")
    print(f"Vocoder = {vocoder_file if vocoder_file else 'Griffin-Lim'}")
    print(f"===================================")

    am_path = Path(am_file)
    if am_path.is_dir():
        meta = am_path / "meta.yaml"
        kwargs = get_dict_from_cache(meta=meta)
    else:    
        kwargs = unpack(input_archive = am_file, outpath = cache_dir)
    print (kwargs)
    
    tts = Text2Speech.from_pretrained(
        vocoder_file=vocoder_file, 
        device=device,
        **kwargs
    )
    return tts

## workaround: fix parallel_wavegan for python3.12
import scipy.signal.windows
import sys
sys.modules['scipy.signal'].kaiser = scipy.signal.windows.kaiser     

print ("ESPnet inicializuotas")    

### 1. Modelis

Nurodykite kelią iki Akustinio modelio ir Vokoderio

In [ ]:
# Nurodykite kelią iki akustinio modelio zip failo
# arba iki direktorijos, jei parsisiuntėte iš HuggingFace
# pvz.:  git clone https://huggingface.co/VSSA-SDSA/sing-agn.fastspeech2.v01
am_zip_file_or_dir="./44test/ner.tts_train_fastspeech2_raw_phn_none_train.loss.ave.44.zip"

# Nurodykite kelią iki vokoderio tar gz failo
# arba iki direktorijos, jei parsisiuntėte iš HuggingFace
# pvz.: git clone https://huggingface.co/VSSA-SDSA/sing-agn.vocoder.style_melgan.v01

# arba palikite tuščią, jei norite naudoti Griffin-Lim algoritmą
vocoder_tar_gz_file_or_dir="./44test/ner.style.v01-44-620000.tar.gz"
# vocoder_tar_gz_file_or_dir="sing-agn.vocoder.style_melgan.v01"
# vocoder_tar_gz_file_or_dir=None

# bus išskleisti į './.cache/' katalogą"
print (f"AM zip failas  : {am_zip_file_or_dir}")
print (f"Vocoder failas : {vocoder_tar_gz_file_or_dir}")

# Inicializuokite TTS modelį
tts = init_tts(am_file=am_zip_file_or_dir, vocoder_file=vocoder_tar_gz_file_or_dir)
print (f"\n\nREADY: modelis įkeltas\n") 

### 2. Sintezavimas

Pakoreguokite/įveskite sakinius, kuriuos norite perskaityti

In [ ]:
texts = ["sil \"i S p r a dZ' ^iu: b \"u v o: v' \"ie n. sp \"a: m.' Z' i n a s , b' e r' \"i b' i s , t a m. s \"u s x a \"o s a s sp g' i: v' \"i: b' E: S a l.' t' \"i n' i s . sil",
        "sil v' \"i s o: S' \"i t o: s v ai z d ^u: , m' i n.' tS' ^iu: sp i ^r.' j eu s m ^u: n \"uo t r u p o: s sp t ^ai s u s m u l.' k' \"E: d a v o: , t ^ai v' ^E: l. i S \"au g d a v o: ^i: j \"eu d' i n a n.' tS' e z d r a m ^a: t' i S k a s' ts' e n \"a s . sil b' \"e ^a: b' e j io: , l' \"iu ts' E: s p a d a r' \"i: t a s' \"i: s p u: d' i s sp b \"u v o: s' t' i p' r' \"eu s' e s sp \"i S' v' \"i s o: a t \"o: s t o: g u: sp g' i: v' ^e: n' i m o: . sil",
        "sil b' \"e t ^o: , tS' \"e b ^o: d \"a: r. sp l' \"iu d o: d r ^au g o: k l' ^ie r' i k o: p' e t' r' ^i: l o: s t' \"E: v' i S' k' E: . sil n u v a Z' ^e: v o: j ^ie d u \"i S v ^a: k a r o: sp ^i: p' i r. m \"uo s' iu s m' i S p a r \"u s . sil",
        "sil t ^uo m' e t \"u S' \"i s k l \"au s' i m a z b \"u v o: l a b ^ai a S t r \"u s , i ^r. p a tS' ^io: s' e k l' e b o: n' \"i j io: s' e t a ^r. p k u n' i g ^u: k' \"i l. d a v o: sp i: v ai r' ^iu: i n.' ts' i d' e ^n. t u: . sil", 
        "sil k l' ^ie r' i k a s v a s ^a: r' i s t ^ai p' i ^r. p a s' i l' \"i k o: b' e s t o: v' ^i: s , s u p l \"o: t a z g' \"E: d o: s' i ^r. s u n' ^ie k' i n. t a z d' ^e: S' i m. t k \"a: r. t u: d au g' ^eu n' e g \"u a n ^uo m' e t , p' e ^r. ^a: t l ai d u s . sil S' \"i t o: k' io: s a p' l' i N.' k' \"i: b' E: s t ^ai j \"i s' n' e b \"u v o: n u m ^a: t' e: s , r ^uo Z d a m a s' i s' ^i: a ^n. t r a: j i: s u s' i t' i k' \"i m a: s \"u l' \"iu ts' e . sil"
    ]
for i, text in enumerate(texts):
    print(f"\n==========================================================================\nText = {text}")
    wav = tts(text)["wav"]
    filename = f"output_44_{i}.wav"
    sf.write(filename, wav.cpu().numpy(), tts.fs)
    display(Audio(filename))
